# 02 — Content Consumption Analysis

**Business question:** What are users actually listening to, and for how long?

This notebook examines content demand from multiple angles: genre-level playtime, listener reach, completion by content length, format performance, and acquisition-channel contribution. The goal is to turn raw consumption behavior into practical content strategy signals.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "master_dataset.csv"
df = pd.read_csv(DATA_PATH)

def first_existing(candidates, required=True):
    for column in candidates:
        if column in df.columns:
            return column
    if required:
        raise KeyError(f"None of these columns exist: {candidates}")
    return None

user_col = first_existing(["user_id"])
content_col = first_existing(["content_id"])
session_col = first_existing(["session_id"])
timestamp_col = first_existing(["timestamp", "started_at", "event_timestamp"], required=False)
format_col = first_existing(["content_format", "format"], required=False)
length_col = first_existing(["content_length_minutes", "duration_minutes"], required=False)
device_col = first_existing(["device_type", "device"], required=False)
subscription_col = first_existing(["subscription_type", "subscription_plan"], required=False)
search_col = first_existing(["search_demand_score", "monthly_searches"], required=False)

if timestamp_col:
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
if "play_count" not in df.columns:
    df["play_count"] = 1
if "skip_count" not in df.columns:
    df["skip_count"] = df["skipped"].astype(int) if "skipped" in df.columns else 0
if "is_completed" in df.columns:
    df["is_completed"] = df["is_completed"].astype(bool)

def safe_qcut(series, labels):
    ranked = series.rank(method="first")
    try:
        return pd.qcut(ranked, q=len(labels), labels=labels).astype(int)
    except ValueError:
        return pd.Series(np.ceil(ranked.rank(pct=True) * len(labels)).clip(1, len(labels)).astype(int), index=series.index)

def low_variance_note(column):
    return df[column].nunique(dropna=True) < 2 if column in df.columns else True

print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]:,} columns from {DATA_PATH}")
df.head()

## 1. Genre Depth vs Reach

Total playtime shows where engagement concentrates, while unique listener count shows audience breadth. Comparing both side by side helps distinguish niche-but-deep genres from broad discovery genres.

In [ ]:
genre_summary = (
    df.groupby("genre", dropna=False)
    .agg(total_playtime_minutes=("session_duration_minutes", "sum"), unique_listeners=(user_col, "nunique"))
    .reset_index()
    .sort_values("total_playtime_minutes", ascending=False)
    .head(min(10, df["genre"].nunique()))
)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Total Playtime", "Unique Listeners"))
fig.add_trace(go.Bar(x=genre_summary["total_playtime_minutes"], y=genre_summary["genre"], orientation="h", name="Playtime"), row=1, col=1)
fig.add_trace(go.Bar(x=genre_summary["unique_listeners"], y=genre_summary["genre"], orientation="h", name="Listeners"), row=1, col=2)
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Top Genres: Engagement Depth vs Audience Reach", height=500, showlegend=False)
fig.show()
genre_summary

## 2. Content Length Sweet Spot

Completion rate is a proxy for content-market fit. Binning content duration helps identify the length range where users are most likely to finish what they start.

In [ ]:
if length_col is None:
    raise KeyError("Content length column is required for duration-bin analysis.")

max_length = df[length_col].max()
bin_edges = [0, 10, 25, 45, max(max_length, 45) + 1]
bin_labels = ["0–10 min", "10–25 min", "25–45 min", "45+ min"]
df["content_length_bin"] = pd.cut(df[length_col], bins=bin_edges, labels=bin_labels, include_lowest=True, right=False)
length_summary = (
    df.groupby("content_length_bin", observed=False)
    .agg(completion_rate=("is_completed", "mean"), avg_duration=(length_col, "mean"), plays=("play_count", "sum"))
    .reset_index()
)
optimal_bin = length_summary.sort_values("completion_rate", ascending=False).iloc[0]["content_length_bin"]

fig = px.bar(length_summary, x="content_length_bin", y="completion_rate", text="completion_rate",
             title=f"Completion Rate by Content Length Bin — Optimal: {optimal_bin}",
             labels={"content_length_bin": "Content Length", "completion_rate": "Average Completion Rate"})
fig.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig.update_yaxes(tickformat=".0%")
fig.show()
length_summary

## 3. Format Comparison

Format-level benchmarks help product and content teams understand whether podcasts, music, or livestream-style content are better suited for completion, session depth, or lower skip behavior.

In [ ]:
if format_col is None:
    raise KeyError("Content format column is required for format analysis.")
format_summary = (
    df.groupby(format_col, dropna=False)
    .agg(avg_session_duration=("session_duration_minutes", "mean"), avg_completion=("is_completed", "mean"), avg_skip_count=("skip_count", "mean"), plays=("play_count", "sum"))
    .reset_index()
    .sort_values("avg_session_duration", ascending=False)
)

fig = make_subplots(rows=1, cols=3, subplot_titles=("Avg Session Duration", "Avg Completion", "Avg Skip Count"))
fig.add_trace(go.Bar(x=format_summary[format_col], y=format_summary["avg_session_duration"], name="Duration"), row=1, col=1)
fig.add_trace(go.Bar(x=format_summary[format_col], y=format_summary["avg_completion"], name="Completion"), row=1, col=2)
fig.add_trace(go.Bar(x=format_summary[format_col], y=format_summary["avg_skip_count"], name="Skips"), row=1, col=3)
fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.update_layout(title="Content Format Performance Comparison", height=450, showlegend=False)
fig.show()
format_summary

## 4. Acquisition Channel Playtime Share

Acquisition channel share shows where valuable listening time enters the platform. This helps growth teams compare channels by engagement quality, not only acquisition volume.

In [ ]:
channel_summary = (
    df.groupby("acquisition_channel", dropna=False)
    .agg(total_playtime_minutes=("session_duration_minutes", "sum"), plays=("play_count", "sum"))
    .reset_index()
)
channel_summary["playtime_share"] = channel_summary["total_playtime_minutes"] / channel_summary["total_playtime_minutes"].sum()

fig = make_subplots(rows=1, cols=2, specs=[[{"type":"domain"}, {"type":"xy"}]], subplot_titles=("Playtime Share", "Playtime Minutes"))
fig.add_trace(go.Pie(labels=channel_summary["acquisition_channel"], values=channel_summary["total_playtime_minutes"], hole=0.35), row=1, col=1)
fig.add_trace(go.Bar(x=channel_summary["acquisition_channel"], y=channel_summary["total_playtime_minutes"]), row=1, col=2)
fig.update_layout(title="Discovery / Acquisition Channel Breakdown", height=500, showlegend=False)
fig.show()
channel_summary.sort_values("total_playtime_minutes", ascending=False)

## 💡 Key Finding

Content under 25 min achieves the highest completion rate in this synthetic sample. For a NOICE-style platform, this supports a short-form content strategy: commission more concise episodes, tighten long-form edits, and use shorter content in onboarding recommendations.